# Advanced Feature Engineering & Multi‑Model Blending for Irrigation Need

#### This notebook implements:
##### - Domain‑specific aggregations, feature crosses, cyclical encoding.
##### - LightGBM + XGBoost training with 5‑fold CV, bias tuning, and probability blending.
##### - Pseudo‑labeling to further lift score.
##### - Visualizations for feature importance, predictions, and confidence distribution.

In [1]:
import os
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from sklearn.preprocessing import TargetEncoder
from sklearn.utils.class_weight import compute_sample_weight
import xgboost as xgb
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

# Set style for plots
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("Set2")

In [5]:
# Configuration
SEED = 42
N_FOLDS = 5
TARGET = 'Irrigation_Need'
ID_COL = 'id'
USE_GPU = True

In [6]:
# Paths
TRAIN_PATH = '/home/chotu/Downloads/Irrigation_Submission/data/raw/train.csv'
TEST_PATH = '/home/chotu/Downloads/Irrigation_Submission/data/raw/test.csv'
SAMPLE_PATH = '/home/chotu/Downloads/Irrigation_Submission/data/raw/sample_submission.csv'
# External dataset (optional, uncomment if available)
EXT_PATH = '/home/chotu/Downloads/Irrigation_Submission/data/OG_data/irrigation_prediction.csv'

# 1. Load data

In [7]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample = pd.read_csv(SAMPLE_PATH)

print(f'Train shape: {train.shape}')
print(f'Test shape : {test.shape}')

Train shape: (630000, 21)
Test shape : (270000, 20)


In [8]:
# Map target
label2idx = {'Low': 0, 'Medium': 1, 'High': 2}
idx2label = {v: k for k, v in label2idx.items()}
train[TARGET] = train[TARGET].map(label2idx).astype('int8')
y = train[TARGET].copy()

X_train_raw = train.drop(columns=[TARGET, ID_COL])
X_test_raw = test.drop(columns=[ID_COL])

In [9]:
# External data
if os.path.exists(EXT_PATH):
    ext = pd.read_csv(EXT_PATH)
    ext[TARGET] = ext[TARGET].map(label2idx).astype('int8')
    X_ext = ext.drop(columns=[TARGET])
    print(f'External shape: {ext.shape}')
    use_external = True
else:
    use_external = False


External shape: (10000, 20)


In [10]:
# Categorical columns
cat_cols = ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season',
            'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']
num_cols = ['Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity',
            'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours',
            'Wind_Speed_kmh', 'Field_Area_hectare', 'Previous_Irrigation_mm']


# 2. Feature Engineering

In [11]:
print('Creating features...')

for cat in cat_cols:
    for num in num_cols:
        # Mean
        mean_map = train.groupby(cat)[num].mean().to_dict()
        X_train_raw[f'{cat}_{num}_mean'] = X_train_raw[cat].map(mean_map)
        X_test_raw[f'{cat}_{num}_mean'] = X_test_raw[cat].map(mean_map)
        if use_external:
            X_ext[f'{cat}_{num}_mean'] = X_ext[cat].map(mean_map)
        # Median
        median_map = train.groupby(cat)[num].median().to_dict()
        X_train_raw[f'{cat}_{num}_median'] = X_train_raw[cat].map(median_map)
        X_test_raw[f'{cat}_{num}_median'] = X_test_raw[cat].map(median_map)
        if use_external:
            X_ext[f'{cat}_{num}_median'] = X_ext[cat].map(median_map)
        # Min
        min_map = train.groupby(cat)[num].min().to_dict()
        X_train_raw[f'{cat}_{num}_min'] = X_train_raw[cat].map(min_map)
        X_test_raw[f'{cat}_{num}_min'] = X_test_raw[cat].map(min_map)
        if use_external:
            X_ext[f'{cat}_{num}_min'] = X_ext[cat].map(min_map)
        # Max
        max_map = train.groupby(cat)[num].max().to_dict()
        X_train_raw[f'{cat}_{num}_max'] = X_train_raw[cat].map(max_map)
        X_test_raw[f'{cat}_{num}_max'] = X_test_raw[cat].map(max_map)
        if use_external:
            X_ext[f'{cat}_{num}_max'] = X_ext[cat].map(max_map)

Creating features...


In [12]:
# Feature crosses
important_pairs = [('Soil_Moisture', 'Rainfall_mm'),
                   ('Temperature_C', 'Wind_Speed_kmh'),
                   ('Sunlight_Hours', 'Temperature_C'),
                   ('Organic_Carbon', 'Electrical_Conductivity'),
                   ('Soil_Moisture', 'Temperature_C'),
                   ('Rainfall_mm', 'Previous_Irrigation_mm')]
for a, b in important_pairs:
    X_train_raw[f'{a}_x_{b}'] = X_train_raw[a] * X_train_raw[b]
    X_test_raw[f'{a}_x_{b}'] = X_test_raw[a] * X_test_raw[b]
    if use_external:
        X_ext[f'{a}_x_{b}'] = X_ext[a] * X_ext[b]

In [13]:
# Cyclical encoding
season_order = {'Winter': 1, 'Spring': 2, 'Summer': 3, 'Fall': 4}
growth_order = {'Sowing': 1, 'Vegetative': 2, 'Flowering': 3, 'Harvest': 4}
for df in [X_train_raw, X_test_raw] + ([X_ext] if use_external else []):
    df['Season_num'] = df['Season'].map(season_order)
    df['Crop_Growth_Stage_num'] = df['Crop_Growth_Stage'].map(growth_order)
    for col, max_val in [('Season_num', 4), ('Crop_Growth_Stage_num', 4)]:
        df[f'{col}_sin'] = np.sin(2 * np.pi * df[col] / max_val)
        df[f'{col}_cos'] = np.cos(2 * np.pi * df[col] / max_val)
    df.drop(columns=['Season_num', 'Crop_Growth_Stage_num'], inplace=True)

In [14]:
# Ratio features
X_train_raw['moisture_per_temp'] = X_train_raw['Soil_Moisture'] / (X_train_raw['Temperature_C'] + 1)
X_test_raw['moisture_per_temp'] = X_test_raw['Soil_Moisture'] / (X_test_raw['Temperature_C'] + 1)
if use_external:
    X_ext['moisture_per_temp'] = X_ext['Soil_Moisture'] / (X_ext['Temperature_C'] + 1)


In [16]:
# Target encoding
te = TargetEncoder(target_type='multiclass', smooth='auto', cv=5, random_state=SEED)

if use_external:
    X_te_fit = pd.concat([X_train_raw, X_ext], axis=0).reset_index(drop=True)
    y_te_fit = pd.concat([y, ext[TARGET]], axis=0).reset_index(drop=True)
else:
    X_te_fit = X_train_raw
    y_te_fit = y

# --- FIX: Ensure y is a clean numpy array or series of a specific type ---
# This prevents the 'ambiguous argument types' error in Cython
y_te_fit_clean = y_te_fit.values.flatten() 

# Also ensure your categorical columns are treated as strings or categories
te.fit(X_te_fit[cat_cols].astype(str), y_te_fit_clean)

for df in [X_train_raw, X_test_raw] + ([X_ext] if use_external else []):
    # Match the .astype(str) here as well during transform
    encoded = te.transform(df[cat_cols].astype(str))
    for i, col in enumerate(cat_cols):
        for class_idx in range(3):
            df[f'TE_{col}_class{class_idx}'] = encoded[:, i*3 + class_idx].astype(np.float32)
    df.drop(columns=cat_cols, inplace=True)

TypeError: Function call with ambiguous argument types

In [ ]:
# Handle inf/nan
for df in [X_train_raw, X_test_raw] + ([X_ext] if use_external else []):
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    for col in df.columns:
        if df[col].isnull().any():
            med = X_train_raw[col].median()
            df[col].fillna(med, inplace=True)

In [ ]:
# Drop constant columns
constant_cols = [c for c in X_train_raw.columns if X_train_raw[c].nunique() == 1]
X_train_raw.drop(columns=constant_cols, inplace=True)
X_test_raw.drop(columns=[c for c in constant_cols if c in X_test_raw.columns], inplace=True)
if use_external:
    X_ext.drop(columns=[c for c in constant_cols if c in X_ext.columns], inplace=True)

In [ ]:
# Combine with external data
if use_external:
    X_train_full = pd.concat([X_train_raw, X_ext], axis=0).reset_index(drop=True)
    y_train_full = pd.concat([y, ext[TARGET]], axis=0).reset_index(drop=True)
    sample_weights = compute_sample_weight(class_weight='balanced', y=y_train_full)
else:
    X_train_full = X_train_raw
    y_train_full = y
    sample_weights = compute_sample_weight(class_weight='balanced', y=y_train_full)

X_test = X_test_raw
print(f'Final training features: {X_train_full.shape[1]}')
print(f'Final test features: {X_test.shape[1]}')

Final training features: 261
Final test features: 261


# 3. Model training: XGBoost + LightGBM with CV


In [ ]:
X = X_train_full.copy()
y = y_train_full.copy()
kf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

xgb_oof = np.zeros((len(X), 3), dtype=np.float32)
lgb_oof = np.zeros((len(X), 3), dtype=np.float32)
xgb_test = np.zeros((len(X_test), 3), dtype=np.float32)
lgb_test = np.zeros((len(X_test), 3), dtype=np.float32)

xgb_params = {
    'objective': 'multi:softprob',
    'num_class': 3,
    'n_estimators': 3500,
    'learning_rate': 0.018,
    'max_depth': 5,
    'subsample': 0.7,
    'colsample_bytree': 0.6,
    'reg_alpha': 0.001,
    'reg_lambda': 7.5,
    'gamma': 0.01,
    'tree_method': 'hist',
    'device': 'cuda' if USE_GPU else 'cpu',
    'random_state': SEED,
    'n_jobs': -1,
    'early_stopping_rounds': 50,
    'eval_metric': 'mlogloss'
}

lgb_params = {
    'objective': 'multiclass',
    'num_class': 3,
    'n_estimators': 3500,
    'learning_rate': 0.02,
    'max_depth': 5,
    'num_leaves': 31,
    'subsample': 0.7,
    'colsample_bytree': 0.6,
    'reg_alpha': 0.01,
    'reg_lambda': 10.0,
    'min_child_samples': 20,
    'random_state': SEED,
    'n_jobs': -1,
    'verbose': -1
}

print('\nStarting 5‑fold CV for XGBoost & LightGBM...')
fold_scores = {'xgb': [], 'lgb': []}
for fold, (tr_idx, val_idx) in enumerate(kf.split(X, y)):
    print(f'Fold {fold+1}/{N_FOLDS}')
    X_tr = X.iloc[tr_idx]
    y_tr = y.iloc[tr_idx]
    w_tr = sample_weights[tr_idx]
    X_val = X.iloc[val_idx]
    y_val = y.iloc[val_idx]

    # XGBoost
    model_xgb = xgb.XGBClassifier(**xgb_params)
    model_xgb.fit(X_tr, y_tr, sample_weight=w_tr,
                  eval_set=[(X_val, y_val)], verbose=False)
    xgb_oof[val_idx] = model_xgb.predict_proba(X_val)
    xgb_test += model_xgb.predict_proba(X_test) / N_FOLDS
    fold_scores['xgb'].append(balanced_accuracy_score(y_val, xgb_oof[val_idx].argmax(axis=1)))

    # LightGBM
    model_lgb = lgb.LGBMClassifier(**lgb_params)
    model_lgb.fit(X_tr, y_tr, sample_weight=w_tr,
                  eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
    lgb_oof[val_idx] = model_lgb.predict_proba(X_val)
    lgb_test += model_lgb.predict_proba(X_test) / N_FOLDS
    fold_scores['lgb'].append(balanced_accuracy_score(y_val, lgb_oof[val_idx].argmax(axis=1)))

    del model_xgb, model_lgb
    gc.collect()



Starting 5‑fold CV for XGBoost & LightGBM...
Fold 1/5


### Fold‑wise balanced accuracy

In [ ]:

plt.figure(figsize=(10,5))
plt.plot(range(1, N_FOLDS+1), fold_scores['xgb'], marker='o', label='XGBoost')
plt.plot(range(1, N_FOLDS+1), fold_scores['lgb'], marker='s', label='LightGBM')
plt.xlabel('Fold')
plt.ylabel('Balanced Accuracy')
plt.title('Fold‑wise Balanced Accuracy (Validation)')
plt.legend()
plt.grid(True)
plt.show()

### Bias tuning

In [ ]:
def tune_bias(proba, y_true):
    best_bias = np.zeros(3)
    best_score = balanced_accuracy_score(y_true, proba.argmax(axis=1))
    for step in [1.0, 0.5, 0.2, 0.1, 0.05, 0.02, 0.01]:
        improved = True
        while improved:
            improved = False
            for c in range(3):
                for d in (-step, step):
                    new_bias = best_bias.copy()
                    new_bias[c] += d
                    adj = np.log(proba + 1e-15) + new_bias
                    pred = np.argmax(adj, axis=1)
                    score = balanced_accuracy_score(y_true, pred)
                    if score > best_score + 1e-9:
                        best_bias, best_score, improved = new_bias, score, True
    return best_bias, best_score

print('\nTuning biases...')
xgb_bias, xgb_ba = tune_bias(xgb_oof, y)
lgb_bias, lgb_ba = tune_bias(lgb_oof, y)
print(f'XGBoost: bias={xgb_bias}, OOF BA={xgb_ba:.6f}')
print(f'LightGBM: bias={lgb_bias}, OOF BA={lgb_ba:.6f}')

def apply_bias(proba, bias):
    adj = np.log(proba + 1e-15) + bias
    return np.exp(adj) / np.exp(adj).sum(axis=1, keepdims=True)

xgb_test_biased = apply_bias(xgb_test, xgb_bias)
lgb_test_biased = apply_bias(lgb_test, lgb_bias)

# 4. Blend probabilities

In [ ]:
# Simple equal weights (you can tune based on OOF BA)
w_xgb = 0.5
w_lgb = 0.5
blended_test = w_xgb * xgb_test_biased + w_lgb * lgb_test_biased
final_preds = blended_test.argmax(axis=1)
final_labels = [idx2label[p] for p in final_preds]

sub = sample.copy()
sub[TARGET] = final_labels
sub.to_csv('blended_submission.csv', index=False)
print('\n✅ Blended submission saved as blended_submission.csv')


### Confusion matrix of OOF predictions (after bias tuning)

In [ ]:
xgb_preds_tuned = apply_bias(xgb_oof, xgb_bias).argmax(axis=1)
lgb_preds_tuned = apply_bias(lgb_oof, lgb_bias).argmax(axis=1)
blend_oof = (w_xgb * apply_bias(xgb_oof, xgb_bias) + w_lgb * apply_bias(lgb_oof, lgb_bias)).argmax(axis=1)

fig, axes = plt.subplots(1, 3, figsize=(15,4))
for ax, preds, title in zip(axes, [xgb_preds_tuned, lgb_preds_tuned, blend_oof],
                              ['XGBoost (tuned)', 'LightGBM (tuned)', 'Blended (tuned)']):
    cm = confusion_matrix(y, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Low','Medium','High'], yticklabels=['Low','Medium','High'])
    ax.set_title(title)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
plt.tight_layout()
plt.show()

### Feature importance from XGBoost (final model)

In [ ]:
# Train a final XGBoost on full data (without CV) to get feature importances
# Remove early_stopping_rounds and eval_metric for full training
final_params = {k: v for k, v in xgb_params.items() if k not in ['early_stopping_rounds', 'eval_metric']}
final_xgb = xgb.XGBClassifier(**final_params)
final_xgb.fit(X, y, sample_weight=sample_weights, verbose=False)
importance = final_xgb.feature_importances_
feat_imp = pd.DataFrame({'feature': X.columns, 'importance': importance}).sort_values('importance', ascending=False).head(30)

plt.figure(figsize=(10,8))
sns.barplot(data=feat_imp, y='feature', x='importance', palette='viridis')
plt.title('Top 30 Feature Importances (XGBoost)')
plt.tight_layout()
plt.show()

# 5. Pseudo‑labeling

In [ ]:
confidence_threshold = 0.995
max_probs = np.max(blended_test, axis=1)
high_conf_idx = max_probs > confidence_threshold
n_pseudo = np.sum(high_conf_idx)
print(f'\nAdding {n_pseudo} pseudo‑labeled samples (confidence > {confidence_threshold})')

# Visualization 4: Confidence distribution
plt.figure(figsize=(8,4))
plt.hist(max_probs, bins=50, alpha=0.7, color='skyblue', edgecolor='black')
plt.axvline(confidence_threshold, color='red', linestyle='--', label=f'threshold = {confidence_threshold}')
plt.xlabel('Max predicted probability')
plt.ylabel('Number of test samples')
plt.title('Confidence distribution of blended predictions')
plt.legend()
plt.show()

In [ ]:
if n_pseudo > 0:
    pseudo_X = X_test.iloc[high_conf_idx].copy()
    pseudo_y = final_preds[high_conf_idx]

    X_aug = pd.concat([X, pseudo_X], axis=0).reset_index(drop=True)
    y_aug = np.concatenate([y, pseudo_y])
    w_aug = compute_sample_weight(class_weight='balanced', y=y_aug)

    print('Retraining XGBoost with augmented data...')
    # Remove early_stopping_rounds and eval_metric for final fit (no validation set)
    final_params = {k: v for k, v in xgb_params.items() 
                    if k not in ['early_stopping_rounds', 'eval_metric']}
    final_model = xgb.XGBClassifier(**final_params)
    final_model.fit(X_aug, y_aug, sample_weight=w_aug, verbose=False)
    pseudo_test_probs = final_model.predict_proba(X_test)

    # Apply bias tuning again (reuse xgb_bias or retune – we reuse for simplicity)
    pseudo_test_biased = apply_bias(pseudo_test_probs, xgb_bias)
    pseudo_final = pseudo_test_biased.argmax(axis=1)
    pseudo_labels = [idx2label[p] for p in pseudo_final]

    sub_pseudo = sample.copy()
    sub_pseudo[TARGET] = pseudo_labels
    sub_pseudo.to_csv('pseudo_submission.csv', index=False)
    print('✅ Pseudo‑labeling submission saved as pseudo_submission.csv')
else:
    print('No high‑confidence samples found – try lowering confidence_threshold.')